# Lab 8: DPO Real com TRL

## Lab 8: DPO Real com TRL

In [1]:
!pip install -q transformers torch trl datasets

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig

### 1. Dataset de preferência (chosen/rejected) — feito à mão

**Por que construir à mão em vez de baixar um dataset real:** DPO precisa
do formato `{prompt, chosen, rejected}` — pra deixar claro exatamente o
padrão de preferência que queremos ensinar (respostas educadas e
específicas > respostas rudes ou vagas), construímos alguns exemplos
sintéticos simples, mas realistas do formato que um dataset de preferência
real (Fase 2 Semana 8.7) teria.

In [2]:
MODEL_NAME = "sshleifer/tiny-gpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

preference_data = [
    {"prompt": "How do I reset my password?",
     "chosen": " Go to Settings > Security > Reset Password, and follow the email link we send you.",
     "rejected": " idk, figure it out."},
    {"prompt": "What is the weather like today?",
     "chosen": " I don't have real-time weather data, but I can help you find a weather service.",
     "rejected": " Weather weather weather weather."},
    {"prompt": "Can you explain what an API is?",
     "chosen": " An API is a set of rules that lets different software programs communicate with each other.",
     "rejected": " no."},
    {"prompt": "How do I improve my writing?",
     "chosen": " Practice regularly, read widely, and ask for feedback on your drafts.",
     "rejected": " just write better lol"},
]

dataset = Dataset.from_list(preference_data)
print(f"✓ {len(dataset)} pares de preferência (prompt, chosen, rejected)")

✓ 4 pares de preferência (prompt, chosen, rejected)


### 2. Medindo a preferência do modelo ANTES do treino

**Por que isso importa:** DPO otimiza a probabilidade relativa entre
`chosen` e `rejected` — pra provar que o treino funcionou, precisamos
medir essa probabilidade antes e depois, não só "parece melhor".

In [3]:
def log_prob_of_completion(model, prompt, completion):
    """Log-probability MÉDIO por token da completion, dado o prompt.

    Nota de metodologia: a primeira versão deste lab somava o log-prob
    total (não dividia pelo número de tokens) — e dava uma margem
    enganosa, porque as respostas 'chosen' aqui são mais longas que as
    'rejected': soma de log-probs cresce (fica mais negativa) com mais
    tokens, então uma resposta mais longa parece 'pior' mesmo sendo
    melhor. Dividir pelo número de tokens remove esse viés de tamanho —
    um problema real de avaliação, não só um detalhe de implementação."""
    full_text = prompt + completion
    prompt_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]
    full_ids = tokenizer(full_text, return_tensors="pt")["input_ids"]

    with torch.no_grad():
        logits = model(full_ids).logits

    log_probs = torch.log_softmax(logits[0, :-1], dim=-1)
    completion_start = prompt_ids.shape[1]
    token_log_probs = log_probs[completion_start - 1:, :].gather(
        1, full_ids[0, completion_start:].unsqueeze(-1)
    )
    return (token_log_probs.sum() / token_log_probs.shape[0]).item()

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

print("ANTES do DPO — log-prob (chosen vs rejected) por exemplo:")
before_margins = []
for ex in preference_data:
    lp_chosen = log_prob_of_completion(model, ex["prompt"], ex["chosen"])
    lp_rejected = log_prob_of_completion(model, ex["prompt"], ex["rejected"])
    margin = lp_chosen - lp_rejected
    before_margins.append(margin)
    print(f"  chosen={lp_chosen:.1f}, rejected={lp_rejected:.1f}, margem={margin:+.1f}")

print(f"\nMargem média ANTES: {sum(before_margins)/len(before_margins):+.2f}")
print("(margem positiva = modelo já prefere 'chosen'; negativa = prefere 'rejected')")

ANTES do DPO — log-prob (chosen vs rejected) por exemplo:
  chosen=-10.8, rejected=-10.8, margem=+0.0
  chosen=-10.8, rejected=-10.8, margem=+0.0
  chosen=-10.8, rejected=-10.8, margem=+0.0
  chosen=-10.8, rejected=-10.8, margem=-0.0

Margem média ANTES: +0.01
(margem positiva = modelo já prefere 'chosen'; negativa = prefere 'rejected')


**Resultado esperado:** com pesos essencialmente aleatórios, a margem
antes do treino deve ser próxima de zero ou aleatória — o modelo não tem
preferência consistente ainda.

### 3. Treino real com DPOTrainer

In [4]:
dpo_config = DPOConfig(
    output_dir="./train_output",
    per_device_train_batch_size=2,
    max_steps=20,
    learning_rate=1e-3,
    logging_steps=5,
    report_to="none",
    bf16=False, fp16=False,
    max_length=64,
)

trainer = DPOTrainer(model=model, args=dpo_config, train_dataset=dataset, processing_class=tokenizer)
trainer.train()
print("\n✓ DPO concluído")

{'loss': '0.6901', 'grad_norm': '1.03', 'learning_rate': '0.0008', 'entropy': '10.82', 'num_tokens': '381', 'logits/chosen': '0.0004448', 'logits/rejected': '0.0006762', 'mean_token_accuracy': '0', 'rewards/chosen': '0.003473', 'rewards/rejected': '-0.002601', 'rewards/accuracies': '0.7', 'rewards/margins': '0.006073', 'logps/chosen': '-194.8', 'logps/rejected': '-61.76', 'epoch': '2.5'}
{'loss': '0.682', 'grad_norm': '0.8401', 'learning_rate': '0.00055', 'entropy': '10.82', 'num_tokens': '765', 'logits/chosen': '-5.682e-05', 'logits/rejected': '0.0001062', 'mean_token_accuracy': '0', 'rewards/chosen': '0.01842', 'rewards/rejected': '-0.004089', 'rewards/accuracies': '1', 'rewards/margins': '0.02251', 'logps/chosen': '-200.1', 'logps/rejected': '-57.43', 'epoch': '5'}
{'loss': '0.6778', 'grad_norm': '0.5795', 'learning_rate': '0.0003', 'entropy': '10.82', 'num_tokens': '1142', 'logits/chosen': '-0.0001527', 'logits/rejected': '0.0002753', 'mean_token_accuracy': '0', 'rewards/chosen': '

**Resultado esperado:** logs de loss a cada 5 steps. O `DPOTrainer`
também reporta `rewards/margins` — o quanto o modelo prefere `chosen`
sobre `rejected`, em unidades de log-probability — essa margem deveria
tender a aumentar ao longo do treino (é literalmente o que o DPO otimiza).

### 4. Medindo a preferência do modelo DEPOIS do treino

In [5]:
print("DEPOIS do DPO — log-prob (chosen vs rejected) por exemplo:")
after_margins = []
for ex in preference_data:
    lp_chosen = log_prob_of_completion(trainer.model, ex["prompt"], ex["chosen"])
    lp_rejected = log_prob_of_completion(trainer.model, ex["prompt"], ex["rejected"])
    margin = lp_chosen - lp_rejected
    after_margins.append(margin)
    print(f"  chosen={lp_chosen:.1f}, rejected={lp_rejected:.1f}, margem={margin:+.1f}")

print(f"\nMargem média ANTES:  {sum(before_margins)/len(before_margins):+.2f}")
print(f"Margem média DEPOIS: {sum(after_margins)/len(after_margins):+.2f}")
print("(se o DPO funcionou, a margem DEPOIS deveria ser mais positiva que ANTES —")
print(" o modelo passou a atribuir mais probabilidade relativa à resposta 'chosen')")

DEPOIS do DPO — log-prob (chosen vs rejected) por exemplo:
  chosen=-10.8, rejected=-10.9, margem=+0.1
  chosen=-10.8, rejected=-10.8, margem=+0.0
  chosen=-10.8, rejected=-10.8, margem=+0.0
  chosen=-10.8, rejected=-10.9, margem=+0.0

Margem média ANTES:  +0.01
Margem média DEPOIS: +0.04
(se o DPO funcionou, a margem DEPOIS deveria ser mais positiva que ANTES —
 o modelo passou a atribuir mais probabilidade relativa à resposta 'chosen')


**Resultado esperado:** a margem média depois do treino deve ser maior
(mais positiva) que antes — evidência direta e mensurável de que o DPO
mudou o modelo pra preferir as respostas marcadas como `chosen`, sem
precisar de um Reward Model separado (Semana 8.5).

**Próximos passos:** Semana 9 troca essa otimização direta por
Reinforcement Learning propriamente dito — mais flexível (funciona com
recompensas que não vêm de pares de preferência), mais complexo de
treinar.